In [4]:
def estimate_vram_usage(batch_size, current_step_t, input_text_len, dtype="float16"):
    """
    Estimates vRAM usage for LLaVA 1.5-7B at decoding step t.
    
    Args:
        batch_size (int): Number of concurrent requests.
        current_step_t (int): Current generation step (0 for prefill).
        input_text_len (int): Average length of text prompt tokens.
        dtype (str): 'float16' or 'float32'.
    
    Returns:
        float: Estimated vRAM usage in GiB.
    """
    # 1. Constants for LLaVA 1.5-7B
    BYTES_PER_PARAM = 2 if dtype == "float16" else 4
    NUM_LAYERS = 32
    HIDDEN_SIZE = 4096
    IMAGE_TOKENS = 576  # Fixed for LLaVA-1.5 (ViT-L-336px)
    TOTAL_PARAMS = 7.2e9 
    
    # 2. Static Memory (Weights)
    mem_static = TOTAL_PARAMS * BYTES_PER_PARAM
    
    # 3. KV Cache Calculation
    # Size per token = 2 (K+V) * Layers * Hidden * Bytes
    kv_size_per_token = 2 * NUM_LAYERS * HIDDEN_SIZE * BYTES_PER_PARAM
    
    total_seq_len = IMAGE_TOKENS + input_text_len + current_step_t
    mem_kv_cache = batch_size * total_seq_len * kv_size_per_token
    
    # 4. Activation/CUDA Overhead (Heuristic)
    # CUDA context (~800MB) + Temp buffers proportional to batch * hidden
    cuda_overhead = 0.8 * (1024**3) 
    activation_buffer = batch_size * HIDDEN_SIZE * BYTES_PER_PARAM * 6 
    
    mem_total_bytes = mem_static + mem_kv_cache + cuda_overhead + activation_buffer
    
    return mem_total_bytes / (1024**3) # Return in GiB

# Example Usage
n = 8
t = 50 # 50th token being generated
text_len = 100

vram_gb = estimate_vram_usage(n, t, text_len)
print(f"Estimated vRAM at step {t} with batch {n}: {vram_gb:.2f} GiB")

Estimated vRAM at step 50 with batch 8: 17.05 GiB
